# STM32MP157C-DK2 Basics

The following notebook explains the basic usage of the STM32MP157C-DK2 development board.

## Preparation

For shell command calls, the Python module ``subprocess`` is used (https://docs.python.org/3/library/subprocess.html).

Run the following code block to provide the functions ``execute_vm_command()`` and ``execute_stm32_command()`` within this notebook.

In [1]:
import subprocess

def execute_vm_command(cmd):
    run = subprocess.run(cmd, universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)
    
def execute_stm32_command(cmd):
    ssh = 'sshpass -p "root" ssh root@192.168.13.13 '
    run = subprocess.run((ssh + cmd), universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)

Of course, all commands given in this tutorial can also be executed directly on within a Linux console.

If you want to open a terminal window, execute the next code block.

In [ ]:
execute_vm_command('gnome-terminal')

---

## Bundle of Components

Check, if you have all of the following components available before you start:

* STM32MP157C-DK2 development board including display
* USB 3.0 cable (Type A to Type C)
* 16 GB micro SD card
* Rasperry Pi USB-C power supply
* USB micro SD card reader
* USB Ethernet adapter
* Ethernet cable
* USB cable (Type A to Micro B)

## Board Interfaces and Hardware Setup

The provided STM32MP157C-DK2 development board has a variety of interfaces as shown in the following figures.

![STM32MP157C-DK2 Board Overview](attachment:stm32mp157c-dk2_overview.png "STM32MP157C-DK2 Board Overview")

These interfaces are relevant for the basic usage of the board:

**Top View Left Side:**

* micro SD card is inserted into micro SD card slot
* USB-C power supply is connected to USB-C connector
* Ethernet cable is connected to Ethernet jack

**Top View Bottom Side:**

* USB cable (Micro B side) is connected to micro USB connector "ST-LINK"

**PC Side:**

* USB Ethernet adapter is connected to PC USB port and to Ethernet cable
* USB cable (Type A side) is connected to PC USB port

**User I/O Interfaces:**

* TFT touch display
* User LEDs for simple output (top view bottom side: "LD5", "LD6", "LD7", "LD8")
* User buttons for simple input (top view bottom side: "USER1" and "USER2")
* Reset button for hard reboot of the system, e.g. in case of a freeze (top view bottom side: "RESET")

---

## Flashing a Micro SD Card Image

The first step in the software setup of this embedded system is the flashing of an image to the micro SD card.

**Preparations for Firmware Flashing:**

* Download the current firmware image for the STM32MP157C-DK2 board from the link provided in the Moodle course and save it to a new directory ``Images`` in your home directory.

In [4]:
execute_vm_command('ls -l ~/Images/*.raw')

-rw-r--r-- 1 student student 2147483648 Oct 20  2025 /home/student/Images/FlashLayout_sdcard_stm32mp157f-dk2-extensible.AB.20251020.raw



Another requirement for the flashing process is the Linux device name of your micro SD card.

First, Make sure the SD card reader is **not** connected to your VM.

Open a terminal window.

In [6]:
execute_vm_command('gnome-terminal')

Then, inside the terminal window execute the command ``lsblk`` to list all currently available block devices.

Keep the terminal window open.

Now, plug in your SD card reader and connect it to your VM, e.g. ``Devices -> USB -> Genesys Logic, Inc. USB Storage`` for USB2.0 readers or ``Devices -> USB -> TS-RDF5A Transcend`` for USB3.0 readers.

Execute ``lsblk`` again.

If everything went fine, you should see an additional block device, e.g. ``sdb`` (i.e. ``/dev/sdb``), that corresponds to your micro SD card.

**Remark:**<br>
This is a *temporary* device, i.e., if you unplug it and replug it later, it might get another name and you have to figure the name out, again.

Afterwards, the tool ``dd`` can be used to copy the image to the micro SD card.

However, access to the micro SD card device requires further permissions, i.e. you have to work with ``sudo``, which requires entering your user password. However, this is not possible or at least inconvenient within Jupyter notebooks. Therefore, the following code block opens a Linux terminal window, in which you can perform all further operations.

In [1]:
execute_vm_command('gnome-terminal')

NameError: name 'execute_vm_command' is not defined

In the terminal window, perform the following two steps:

1. Unmount all mounted partitions of your micro SD card device (replace *X* by your device)<br>
   ``sudo umount /dev/sdX*``<br><br>
2. Copy the available image to your micro SD card (replace *X* by your device)<br>
   ``sudo dd if=<your-image> of=/dev/sdX bs=16M status=progress``<br><br>
   (Example: ``sudo dd if=../../Images/stm32mp157c.raw of=/dev/sdb bs=16M status=progress``)<br><br>
3. Use ``sync`` to write any data buffered in memory out to disk.<br>

**!!! Caution !!!**<br>
If you choose the wrong device, ``dd`` might overwrite it and thereby break your whole system. Double check the parameters of ``dd``!

After ``dd`` has finished (might take several minutes), your micro SD card is prepared to boot.

**!!! Caution !!!**<br>
Remove your micro SD card / SD card reader *safely* !!! Otherwise, the data might get corrupted.

Insert it into the micro SD card slot on the development board!

---

## Connecting to the Board via UART

The Universal Asynchronous Receiver Transmitter (UART) is a popular serial communication interface in embedded systems. It can be used for various applications, e.g. to provide developers with boot and debug messages from an embedded system.

**Preparations for UART Communication:**

* Connect the micro USB cable to your board and your PC
* Pass the USB device through to your VM, e.g. ``Devices -> USB -> STMicroelectronics STM32 STLink``
* Make sure the power supply is **not yet connected**

Afterwards, check if your UART device is available in your Linux system (probably as ``/dev/ttyACM0``).

In [5]:
execute_vm_command('ls /dev/tty* | grep ACM')

NameError: name 'execute_vm_command' is not defined

Now, open a new terminal window and connect to the board's UART interface with the tool ``picocom`` and the following command (replace ``/dev/ttyACM0`` with your UART device, if necessary):

``sudo picocom /dev/ttyACM0 -b 115200``

In [ ]:
execute_vm_command('gnome-terminal')

**Boot and Login:**

* Plug in the power supply and watch the STM32MP157C-DK2 board boot.
* At the end of the boot process, there should be a login prompt in your terminal window and also on the board's TFT display.
* You can login to the system with the user ``root`` and the password ``root``.

Finally, you are now able to perform operation directly on the embedded system.

---

## Connecting to the Board via SSH

While a UART connection is important to get boot and debug messages, it is not the most convenient way to work with an embedded system.

A network connection over Secure Shell (SSH) is helpful in many ways.

**Preparations for SSH Communication:**

* Connect the USB Ethernet adapter and the Ethernet cable to your board and your PC
* Pass the USB device through to your VM, e.g. ``Devices -> USB -> ASIX Elec. Corp. AX88772B``

Afterwards, open a new terminal window to configure and test your USB Ethernet adapter.

In [ ]:
execute_vm_command('gnome-terminal')

Perform these steps to get your USB Ethernet adapter operational:

1. Find the name of your USB Ethernet adapter device in Linux<br>
   ``ip link show`` (probably the last in the list, e.g. ``eth1``)<br><br>
2. Activate the interface<br>
   ``sudo ip link set up dev <your-ethernet-device>``<br><br>
3. Set the IP address of the interface<br>
   ``sudo ip addr add 192.168.13.17/24 dev <your-ethernet-device>``<br><br>
4. Try to reach the STM32MP157C-DK2 board by its statically configured IP address ``192.168.13.13/24``<br>
   ``ping -I <your-ethernet-device> -c 4 192.168.13.13``<br><br>

Finally, you can use the following command to connect to your device by SSH:

``ssh root@192.168.13.13``

The password is ``root`` (same user/password as for terminal login over UART).

**Troubleshooting / SSH Key Reset:**

Sometimes, e.g. in case of using a new firmware image on the STM32MP157C-DK2 board, it might be necessary to remove the previously used SSH key from the ``known_hosts`` file in the VM to enable SSH connections with the board again.

In [ ]:
execute_vm_command('ssh-keygen -f "/home/student/.ssh/known_hosts" -R "192.168.13.13"')

Execute the following command to check, if the SSH connection works as intended. This saves the new SSH key in your ``known_hosts`` file.

In [ ]:
execute_vm_command('sshpass -p "root" ssh -o StrictHostKeyChecking=accept-new root@192.168.13.13 uname -a')

If everything worked as intended, an output similar to the following one should be printed:

```
Linux stm32mp1 5.15.67 #1 SMP PREEMPT ...
```

## Copying Files from and to the Device via SCP

If connecting to the board via SSH works without problems, the SSH-based Secure Copy (SCP) protocol can be used to copy files from and to the device.

Since, again, password entry will be necessary, perform the following commands within a terminal window.

In [ ]:
execute_vm_command('gnome-terminal')

The subsequent command copies this Jupyter notebook file to the home directory ``/home/root`` of the user ``root`` on the device.

``scp NB01_STM32MP157C-DK2_Basics.ipynb root@192.168.13.13:/home/root``

On the other hand, this command pulls the Markdown file at ``/home/lservice/repair_notes.md`` from the device and stores it in the current directory (``.``) in this VM.

``scp root@192.168.13.13:/home/lservice/repair_notes.md .``

**Hint:**<br>
If you want to copy whole directories from or to the device, use the option `-r` for recursive copying of all contained files.